# EVOLVE — Training YOLOv8s under Extreme Concert Conditions

This notebook trains and compares two object detection models on the EVOLVE dataset:

1. YOLOv8s initialized with COCO-pretrained weights (transfer learning)
2. YOLOv8s trained from random initialization (scratch)

Both models are trained using identical data splits, hyperparameters, and random seeds to ensure a controlled comparison.

## 1. Motivation

Most object detection models are trained on standard datasets such as COCO, which primarily contain well-lit natural images.

Concert environments differ in illumination conditions, crowd density, motion blur, and occlusion patterns.

This notebook evaluates whether transfer learning from COCO improves detection performance in this domain compared to training from scratch.

## 2. Experimental Setup

This section defines:

- Dataset configuration and split
- Model architecture (YOLOv8s)
- Training hyperparameters
- Reproducibility settings (random seed and environment)

Both training regimes use identical experimental conditions to ensure a controlled comparison.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/rinarazafimahefa/Documents/SISE/Computer_Vision/evolve")

In [ ]:
# ==============================
# Imports
# ==============================

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO


# ==============================
# Reproducibility
# ==============================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# If CUDA is available, also seed CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Make algorithms more deterministic where possible
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# ==============================
# Device configuration
# ==============================

device = "cpu"

print("Device used:", device)
print("PyTorch version:", torch.__version__)

In [ ]:
# ==============================
# Dataset configuration
# ==============================

from pathlib import Path
import yaml

DATASET_CONFIG_PATH = Path("../configs/dataset.yaml").resolve()

with open(DATASET_CONFIG_PATH, "r", encoding="utf-8") as f:
    dataset_config = yaml.safe_load(f)

# Resolve dataset root RELATIVE TO the YAML file location (configs/)
BASE_PATH = (DATASET_CONFIG_PATH.parent / dataset_config["path"]).resolve()

train_path = BASE_PATH / dataset_config["train"]
val_path   = BASE_PATH / dataset_config["val"]
test_path  = BASE_PATH / dataset_config.get("test", "images/test")

print("Dataset root:", BASE_PATH)
print("Train directory:", train_path)
print("Validation directory:", val_path)
print("Test directory:", test_path)

# Check existence
assert train_path.exists(), f"Train directory does not exist: {train_path}"
assert val_path.exists(), f"Validation directory does not exist: {val_path}"
assert test_path.exists(), f"Test directory does not exist: {test_path}"

# Class names
names = dataset_config["names"]
if isinstance(names, dict):
    # enforce numeric order by key
    class_names = [names[k] for k in sorted(names)]
else:
    class_names = list(names)

print("Number of classes:", len(class_names))
print("Class names:", class_names)

In [ ]:
# ==============================
# Dataset volume verification
# ==============================

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

train_images = [
    p for p in train_path.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
]

val_images = [
    p for p in val_path.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
]

print("Number of training images:", len(train_images))
print("Number of validation images:", len(val_images))

In [ ]:
# ==============================
# Class distribution analysis (training set)
# ==============================

from collections import defaultdict

label_train_path = BASE_PATH / "labels" / "train"
assert label_train_path.exists(), "Training label directory does not exist."

class_counts = defaultdict(int)

label_files = list(label_train_path.glob("*.txt"))

for label_file in label_files:
    with open(label_file, "r") as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

# Ensure all classes appear (even if count = 0)
distribution = {}

for class_id, class_name in enumerate(class_names):
    distribution[class_name] = class_counts.get(class_id, 0)

print("Class distribution (training set):")
for k, v in distribution.items():
    print(f"{k}: {v}")

In [ ]:
# ==============================
# Class distribution visualization
# ==============================

import matplotlib.pyplot as plt

# Sort classes by descending frequency (more informative)
sorted_items = sorted(distribution.items(), key=lambda x: x[1], reverse=True)

classes = [k for k, _ in sorted_items]
counts = [v for _, v in sorted_items]

plt.figure(figsize=(10, 5))
bars = plt.bar(classes, counts)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Classes")
plt.ylabel("Number of instances")
plt.title("Training Set Class Distribution")

# Add value labels above bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        str(height),
        ha="center",
        va="bottom"
    )

plt.tight_layout()

# Save figure for report
os.makedirs("../results/figures", exist_ok=True)
plt.savefig("../results/figures/class_distribution.png")

plt.show()

### Global Dataset Instance Verification

In [ ]:
# ==============================
# Global class distribution (train + val + test)
# ==============================

from collections import defaultdict

label_paths = [
    BASE_PATH / "labels" / "train",
    BASE_PATH / "labels" / "val",
    BASE_PATH / "labels" / "test"
]

global_class_counts = defaultdict(int)

for label_path in label_paths:
    for label_file in label_path.glob("*.txt"):
        with open(label_file, "r") as f:
            for line in f:
                if line.strip():
                    class_id = int(line.split()[0])
                    global_class_counts[class_id] += 1

global_distribution = {}

for class_id, class_name in enumerate(class_names):
    global_distribution[class_name] = global_class_counts.get(class_id, 0)

print("Global class distribution (train + val + test):")
for k, v in global_distribution.items():
    print(f"{k}: {v}")

min_instances = min(global_distribution.values())
print("\nMinimum number of instances in a class:", min_instances)

if min_instances >= 30:
    print("✔ Requirement satisfied: at least 30 instances per class.")
else:
    print("⚠ Warning: Some classes have fewer than 30 instances.")

In [ ]:
# ==============================
# Results directories
# ==============================

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
QUALITATIVE_DIR = RESULTS_DIR / "qualitative_examples"
CONFUSION_DIR = RESULTS_DIR / "confusion_analysis"

for d in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR, QUALITATIVE_DIR, CONFUSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Results directories initialized.")

## 3. Validation Performance Comparison

This section compares validation performance between:

- The pretrained (transfer learning) model
- The scratch (random initialization) model

Global detection metrics (mAP, precision, recall) are reported under identical evaluation settings.

In [ ]:
# ==========================================
# Experimental Configuration
# ==========================================

"""
Common experimental setup for transfer learning comparison.

Two training regimes:
1) COCO-pretrained fine-tuning
2) Random initialization (from scratch)

Both runs use identical hyperparameters to ensure
a controlled and fair comparison.
"""

EPOCHS = 30
IMGSZ = 640
BATCH_SIZE = 16
PATIENCE = 8

PROJECT_DIR = Path("../runs/train").resolve()

print("Experimental configuration loaded.")

In [ ]:
from pathlib import Path
import yaml

DATASET_CONFIG_PATH = Path("../configs/dataset.yaml").resolve()

with open(DATASET_CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

base = Path(cfg["path"])
for split in ["train", "val", "test"]:
    img_dir = base / cfg[split]
    lbl_dir = base / "labels" / split
    print(split, "images:", img_dir, "exists:", img_dir.exists())
    print(split, "labels:", lbl_dir, "exists:", lbl_dir.exists())

In [ ]:
# ==========================================
# Training Regime 1 - Pretrained (COCO)
# ==========================================

"""
Transfer learning experiment.

The model is initialized with COCO-pretrained weights (yolov8s.pt)
and fine-tuned on the EVOLVE dataset.

This setup evaluates the impact of generic visual pretraining
under extreme concert conditions.
"""

model_pretrained = YOLO("yolov8s.pt")

results_pretrained = model_pretrained.train(
    data=DATASET_CONFIG_PATH,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    seed=SEED,
    patience=PATIENCE,
    device=device,
    workers=2,
    project=PROJECT_DIR,
    name="evolve_yolov8s_pretrained",
    exist_ok=True
)

print("Pretrained training completed.")

In [ ]:
# ==============================
# Experimental Setup
# ==============================

from pathlib import Path
import os
import torch
from ultralytics import YOLO
import ultralytics

# --- Notebook location-aware project root ---
# If your notebook is in notebooks/, PROJECT_ROOT = .. (repo root)
PROJECT_ROOT = Path("..").resolve()

# --- Run selection (must match Notebook 1 training) ---
RUN_NAME = "evolve_yolov8s_pretrained"
RUN_DIR = (PROJECT_ROOT / "runs" / "train" / RUN_NAME).resolve()

WEIGHTS_DIR = RUN_DIR / "weights"
LAST_WEIGHTS = WEIGHTS_DIR / "last.pt"
BEST_WEIGHTS = WEIGHTS_DIR / "best.pt"

print("Ultralytics version:", getattr(ultralytics, "__version__", None))
print("Torch version:", torch.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_DIR:", RUN_DIR)
print("LAST_WEIGHTS exists?", LAST_WEIGHTS.exists(), "-", LAST_WEIGHTS)
print("BEST_WEIGHTS exists?", BEST_WEIGHTS.exists(), "-", BEST_WEIGHTS)

# Pick weights: prefer last for resume, else fall back to best
WEIGHTS_TO_LOAD = LAST_WEIGHTS if LAST_WEIGHTS.exists() else BEST_WEIGHTS
assert WEIGHTS_TO_LOAD.exists(), (
    "Neither last.pt nor best.pt found. "
    "Check RUN_NAME and your notebook working directory."
)

model = YOLO(str(WEIGHTS_TO_LOAD))
print("Loaded weights:", WEIGHTS_TO_LOAD)

In [ ]:
# ==========================================
# Training Regime 2 - From Scratch
# ==========================================

"""
Learning without transfer.

The YOLOv8s architecture (yolov8s.yaml) is initialized with random weights and trained exclusively on EVOLVE.

This experiment isolates the contribution of domain-specific training without prior visual pretraining.
"""

model_scratch = YOLO("yolov8s.yaml")

results_scratch = model_scratch.train(
    data=DATASET_CONFIG_PATH,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    seed=SEED,
    patience=PATIENCE,
    device=device,
    workers=2,
    project=PROJECT_DIR,
    name="evolve_yolov8s_scratch",
    exist_ok=True
)

print("Scratch training completed.")

In [ ]:
from pathlib import Path
import os

print("Current working dir:", Path.cwd())

PROJECT_ROOT = Path.cwd().parent  # remonte de notebooks vers evolve
print("Project root:", PROJECT_ROOT)

best_pretrained_path = PROJECT_ROOT / "runs/train/evolve_yolov8s_pretrained/weights/best.pt"
best_scratch_path = PROJECT_ROOT / "runs/train/evolve_yolov8s_scratch/weights/best.pt"

print("Pretrained exists:", best_pretrained_path.exists())
print("Scratch exists:", best_scratch_path.exists())

print(best_pretrained_path)
print(best_scratch_path)

In [ ]:
# ==========================================
# Validation Comparison
# ==========================================

from ultralytics import YOLO
import pandas as pd

metrics_pretrained = YOLO(str(best_pretrained_path)).val(
    data=str(DATASET_CONFIG_PATH),
    split="val",
    device=device
)

metrics_scratch = YOLO(str(best_scratch_path)).val(
    data=str(DATASET_CONFIG_PATH),
    split="val",
    device=device
)

comparison = pd.DataFrame([
    {
        "Model": "Pretrained (COCO)",
        "mAP50-95": float(metrics_pretrained.box.map),
        "mAP50": float(metrics_pretrained.box.map50),
        "Precision": float(metrics_pretrained.box.mp),
        "Recall": float(metrics_pretrained.box.mr)
    },
    {
        "Model": "Scratch (Random Init)",
        "mAP50-95": float(metrics_scratch.box.map),
        "mAP50": float(metrics_scratch.box.map50),
        "Precision": float(metrics_scratch.box.mp),
        "Recall": float(metrics_scratch.box.mr)
    }
])

delta = comparison.iloc[0, 1:] - comparison.iloc[1, 1:]

display(comparison)
display(delta)

In [ ]:
from pathlib import Path
import json
import shutil
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# Transfer Learning Gain + Unified Export
# ==========================================

metric_cols = ["mAP50-95", "mAP50", "Precision", "Recall"]

# --- Safety checks (fail fast, clearer errors) ---
required_models = {"Pretrained (COCO)", "Scratch (Random Init)"}
missing = required_models - set(comparison["Model"].unique())
if missing:
    raise ValueError(f"Missing model(s) in comparison table: {missing}")

# --- Compute delta ---
cmp = comparison.set_index("Model")[metric_cols]
delta = cmp.loc["Pretrained (COCO)"] - cmp.loc["Scratch (Random Init)"]

print("Transfer learning gain (Pretrained - Scratch):")
try:
    display(delta.to_frame(name="delta"))  # works in notebooks
except NameError:
    print(delta.to_frame(name="delta"))

# Simple verdict (based on mAP50-95)
if delta["mAP50-95"] > 0:
    print("✅ Transfer learning helps on EVOLVE (higher mAP50-95).")
elif delta["mAP50-95"] < 0:
    print("⚠️ Scratch is better on EVOLVE (possible domain mismatch with COCO).")
else:
    print("➖ No difference on mAP50-95.")

# --- Output dirs ---
results_dir = Path("../results")
fig_dir = results_dir / "figures"
results_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# --- Save tables ---
comparison_csv = results_dir / "pretrained_vs_scratch_metrics.csv"
delta_csv = results_dir / "pretrained_vs_scratch_delta.csv"

comparison.to_csv(comparison_csv, index=False)
delta.to_frame(name="delta").to_csv(delta_csv, index=True)

# ==========================================
# Load training logs (both regimes)
# ==========================================

log_pretrained_path = PROJECT_ROOT / "runs/train/evolve_yolov8s_pretrained/results.csv"
log_scratch_path = PROJECT_ROOT / "runs/train/evolve_yolov8s_scratch/results.csv"

print(log_pretrained_path.exists())
print(log_scratch_path.exists())

training_log_pretrained = pd.read_csv(log_pretrained_path)
training_log_scratch = pd.read_csv(log_scratch_path)

print("Pretrained log loaded from:", log_pretrained_path)
print("Scratch log loaded from:", log_scratch_path)

# Helper: plot function
def plot_metric(metric_col, ylabel, title, out_path, label_pre="Pretrained (COCO)", label_scr="Scratch (Random Init)"):
    if metric_col not in training_log_pretrained.columns:
        raise KeyError(f"{metric_col} missing from pretrained log columns.")
    if metric_col not in training_log_scratch.columns:
        raise KeyError(f"{metric_col} missing from scratch log columns.")

    plt.figure(figsize=(8, 5))
    plt.plot(training_log_pretrained["epoch"], training_log_pretrained[metric_col], label=label_pre)
    plt.plot(training_log_scratch["epoch"], training_log_scratch[metric_col], label=label_scr)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()

# --- Figures ---
plot_metric(
    metric_col="val/box_loss",
    ylabel="Validation Box Loss",
    title="Validation Box Loss – Pretrained vs Scratch",
    out_path=fig_dir / "val_box_loss_comparison.png"
)

plot_metric(
    metric_col="metrics/mAP50(B)",
    ylabel="mAP@0.5",
    title="Validation mAP@0.5 – Pretrained vs Scratch",
    out_path=fig_dir / "map50_comparison.png",
    label_pre="Pretrained mAP@0.5",
    label_scr="Scratch mAP@0.5"
)

plot_metric(
    metric_col="metrics/mAP50-95(B)",
    ylabel="mAP@0.5:0.95",
    title="Validation mAP@0.5:0.95 – Pretrained vs Scratch",
    out_path=fig_dir / "map50_95_comparison.png",
    label_pre="Pretrained mAP@0.5:0.95",
    label_scr="Scratch mAP@0.5:0.95"
)

# ==========================================
# Export metadata snapshot (single JSON)
# ==========================================

payload = {
    "experiment": "pretrained_vs_scratch",
    "seed": SEED,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch_size": BATCH_SIZE,
    "patience": PATIENCE,
    "dataset_config_path": str(DATASET_CONFIG_PATH),
    "data_split": {
        "train_images": len(train_images) if "train_images" in globals() else None,
        "val_images": len(val_images) if "val_images" in globals() else None,
    },
    "runs": {
        "pretrained_run_dir": str(PROJECT_ROOT / "runs/train/evolve_yolov8s_pretrained"),
        "scratch_run_dir": str(PROJECT_ROOT / "runs/train/evolve_yolov8s_scratch"),
        "pretrained_best_weights": str(best_pretrained_path),
        "scratch_best_weights": str(best_scratch_path),
        "pretrained_results_csv": str(log_pretrained_path),
        "scratch_results_csv": str(log_scratch_path),
    },
    "metrics_table": comparison.to_dict(orient="records"),
    "delta": delta.to_dict(),
    "env": {
        "torch_version": getattr(torch, "__version__", None),
        "ultralytics_version": getattr(__import__("ultralytics"), "__version__", None),
        "device": device if "device" in globals() else None
    }
}

json_path = results_dir / "pretrained_vs_scratch_summary.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)

print("Exports written:")
print(" -", comparison_csv)
print(" -", delta_csv)
print(" -", json_path)

# ==========================================
# Export best weights
# ==========================================

shutil.copy(best_pretrained_path, results_dir / "best_pretrained.pt")
shutil.copy(best_scratch_path, results_dir / "best_scratch.pt")
print("Exported best_pretrained.pt and best_scratch.pt")

## Conclusion

This notebook compared two training regimes on the EVOLVE dataset:

- YOLOv8s initialized with COCO-pretrained weights
- YOLOv8s trained from scratch

Under identical training conditions, the pretrained model achieved substantially higher validation performance across all detection metrics.

The scratch model struggled to converge effectively under the current dataset size, indicating that transfer learning is necessary when training data is limited and visual conditions differ from standard natural image datasets.

Although transfer learning improves performance, validation results remain moderate, suggesting that additional data or domain-specific training strategies may be required to further improve detection accuracy.